# At times, code will be reaped from the project

### Below we have the manual implementation of the Gravity theory of trade

Used in 02_EDA in section 7.

In [ ]:
# In an attempt to synthesise the missing data in the dataframe, we can use the Gravity formula of trade.
# For more notes, see notes_for_work.md (03/04/2026)
#
# Question: Using pure raw gdp_o and gdp_d leads to very unbalanced synthetic data. Might be worth trying gdpcap_ppp_o


df = ecowas_df_full.copy()

# So to train our model, we must remove all NaNs. Don't want it to learn from what's missing, obviously.
train_df = df.dropna(subset=["tradeflow_baci", "gdpcap_ppp_o", "gdpcap_ppp_d", "dist", "pop_o", "pop_d"])

# Logs for variables included in PPML
train_df["ln_gdp_o"] = np.log(train_df["gdpcap_ppp_o"])
train_df["ln_gdp_d"] = np.log(train_df["gdpcap_ppp_d"])
train_df["ln_dist"]  = np.log(train_df["dist"])
train_df["ln_pop_o"] = np.log(train_df["pop_o"])
train_df["ln_pop_d"] = np.log(train_df["pop_d"])


# We want to include fixed effects. We use the pyfixest library:
# Since we currently do not have fixed effects, this is just a standard Pooled Poisson Pseudo-Maximum Likelihood regression (PPML)
res = fepois(
    fml="tradeflow_baci ~ ln_gdp_o + ln_gdp_d + ln_dist + sibling + ln_pop_o + ln_pop_d",        # Here we can add Fixed Effects by adding a | after the coefficients
    data=train_df,
    )

print(res.summary())



#### BELOW WE HAVE MANUAL IMPLEMENTATION OF GRAVITY TRADE MODEL
# Not necessary, since we have an automatic predict

# Here we can extract the coefficients. 
coefs = res.coef()

beta1 = coefs["ln_gdp_o"]
beta2 = coefs["ln_gdp_d"]
beta3 = -coefs["ln_dist"]  # negative because distance reduces trade

# Below we save the effects of the fixed effects in a dict
#fixef = res.fixef()

# We also need to figure out which fixed effects to include as part of this synthesising of data. 
# These fixed effects should make the synthetic data more precise, by adding other variables to map some of the complexity of the involved countries 
# 
numerator   = train_df["tradeflow_baci"].mean()

denominator = ((train_df["gdp_o"]**beta1) *
               (train_df["gdp_d"]**beta2) /
               (train_df["dist"]**beta3)).mean()

G = numerator / denominator
print(f"Our scaling constant G is: {G}")

df["ln_gdp_o"] = np.log(df["gdp_o"])
df["ln_gdp_d"] = np.log(df["gdp_d"])
df["ln_dist"]  = np.log(df["dist"])

df["synthetic_trade"] = (
    G *
    (df["gdp_o"] ** beta1) *
    (df["gdp_d"] ** beta2) /
    (df["dist"]  ** beta3)
)

# Optional: add noise (gamma noise works well for multiplicative models)
noise = np.random.gamma(shape=1.0, scale=0.15, size=len(df))
df["synthetic_trade_noisy"] = df["synthetic_trade"] * (1 + noise)


